# SIH26072 nowcasting — training on Colab (T4)

Thin wrapper around the package CLIs. Runtime → *Change runtime type* → **T4 GPU**.

Stages: (a) SEVIR pretrain → (b) India fine-tune → (c) lightning head → (d) calibration → eval → export.
Artifacts and reports are written to Google Drive so they survive the session.
Every stage can be re-run on its own; each prints the artifact directory it created.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import sys; print(sys.version)

## 1. Settings

In [ ]:
# Where the code comes from: a git URL, or a folder on Drive that contains `ml/`.
REPO_URL = ""                                        # e.g. "https://github.com/<org>/<repo>.git"
DRIVE_REPO = "/content/drive/MyDrive/sih26072/repo"  # used when REPO_URL is empty
DRIVE_OUT = "/content/drive/MyDrive/sih26072"        # artifacts, reports, data cache

SEVIR_EVENTS = 300        # size of the SEVIR subset to download (cached on Drive)
INDIA_EVENTS = f"{DRIVE_OUT}/data/events"            # India event Zarrs from the data team
BATCH = 16                # T4 16 GB at 128x128 with 16-mixed; lower if OOM
EPOCHS_PRETRAIN, EPOCHS_FINETUNE, EPOCHS_LTG = 30, 20, 10

## 2. Mount Drive and install

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, subprocess
if REPO_URL:
    !git clone -q {REPO_URL} /content/repo
    REPO = "/content/repo"
else:
    !rsync -a --exclude .venv --exclude artifacts --exclude runs {DRIVE_REPO}/ /content/repo/
    REPO = "/content/repo"
%cd {REPO}/ml
# torch is pinned in pyproject.toml; if Colab ships a different version pip replaces it (takes a few minutes).
!pip install -q -e ".[dev,sevir]"

In [ ]:
# Offline test suite (~1 min) + the T4 inference budget test (< 2 s per domain).
!python -m pytest -q
!python -m pytest -q -m gpu tests/integration/test_gpu_benchmark.py -s

## 3. Data

In [ ]:
SEVIR_DIR = f"{DRIVE_OUT}/data/sevir"
if not os.path.exists(f"{SEVIR_DIR}/CATALOG.csv"):
    !python scripts/download_sevir_subset.py --out {SEVIR_DIR} --n-events {SEVIR_EVENTS}
else:
    print("SEVIR subset cached at", SEVIR_DIR)

# Validate the India events against the input contract before training on them.
if os.path.isdir(INDIA_EVENTS):
    !nowcast-validate {INDIA_EVENTS} --no-values
else:
    print("No India events found at", INDIA_EVENTS, "- stages b-d need them.")

## 4. Stage (a): SEVIR pretraining

In [ ]:
REG = f"{DRIVE_OUT}/artifacts/models"
out = !nowcast-train -c configs/train/pretrain_sevir.yaml data.sevir.root={SEVIR_DIR} data.batch_size={BATCH} \
    train.max_epochs={EPOCHS_PRETRAIN} train.output_dir=/content/runs/pretrain registry.root={REG} model.name=nowcast_sevir
print("\n".join(out[-5:]))
ART_A = out[-1]

## 5. Stage (b): India fine-tune (backbone frozen, then unfrozen)

In [ ]:
out = !nowcast-train -c configs/train/finetune_india.yaml data.events_dir={INDIA_EVENTS} data.batch_size={BATCH} \
    train.init_from={ART_A} train.max_epochs={EPOCHS_FINETUNE} train.output_dir=/content/runs/finetune registry.root={REG}
print("\n".join(out[-5:]))
ART_B = out[-1]

## 6. Stage (c): lightning head, (d): isotonic calibration

In [ ]:
out = !nowcast-train -c configs/train/lightning_head.yaml data.events_dir={INDIA_EVENTS} data.batch_size={BATCH} \
    train.init_from={ART_B} train.max_epochs={EPOCHS_LTG} train.output_dir=/content/runs/ltg registry.root={REG}
print("\n".join(out[-5:]))
ART_C = out[-1]
!nowcast-calibrate --model {ART_C} --events {INDIA_EVENTS} --split val

## 6b. Optional stage (e): diffusion refiner for ensemble members

In [ ]:
TRAIN_REFINER = True
if TRAIN_REFINER:
    out = !nowcast-train -c configs/train/refiner.yaml data.events_dir={INDIA_EVENTS} data.batch_size={BATCH} \
        train.init_from={ART_C} train.output_dir=/content/runs/refiner registry.root={REG}
    print("\n".join(out[-5:]))
    ART_C = out[-1]   # evaluate/export the refined artifact (same base model + calibrator + refiner.pt)

## 7. Evaluate on the test split, export, smoke-predict

In [ ]:
REPORT = f"{DRIVE_OUT}/reports/india_test"
!nowcast-eval --model {ART_C} --events {INDIA_EVENTS} --split test --baselines all --out {REPORT}
!nowcast-export --model {ART_C}
import glob
ev = sorted(glob.glob(f"{INDIA_EVENTS}/*.zarr"))[0]
!nowcast-predict --model {ART_C} --event {ev} --t0 auto --out /content/forecast.zarr

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open(f"{REPORT}/skill.md").read().split("## Figures")[0]))
for f in ("csi_vs_lead.png", "reliability.png", "case_study.png"):
    display(Image(f"{REPORT}/{f}"))
print("Model card:", f"{ART_C}/model_card.md")